# Exploratory Data Analysis (EDA) - Módulo 2

Este notebook analiza el comportamiento de los conductores a partir del *Multi-Class Driver Behavior Image Dataset*.
Vamos a expandir el análisis para:
1. Validar la estructura y cantidad de imágenes por clase.
2. Visualizar muestras aleatorias de las distracciones.
3. Concluir sobre los riesgos más frecuentes y proponer medidas preventivas.

In [ ]:
import os
import random
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from PIL import Image

# Configuramos el estilo de las gráficas
sns.set_theme(style="whitegrid")

# Ruta correcta tras la extracción del ZIP
DATA_DIR = '../data/driver_behavior_dataset/Multi-Class Driver Behavior Image Dataset/'

print(f"Verificando directorio: {DATA_DIR}")
if os.path.exists(DATA_DIR):
    print("✅ Directorio encontrado.")
else:
    print("❌ Directorio no encontrado. Verifica la ruta.")

### 1. Distribución de Clases
Analizamos cuántas imágenes tenemos por cada tipo de comportamiento. Es vital saber si el dataset está balanceado.

In [ ]:
classes = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]
class_counts = {}

for c in classes:
    class_path = os.path.join(DATA_DIR, c)
    # Contamos solo archivos que terminen en formato de imagen
    num_images = len([f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    class_counts[c] = num_images

df_counts = pd.DataFrame(list(class_counts.items()), columns=['Comportamiento', 'Cantidad de Imágenes'])
df_counts = df_counts.sort_values(by='Cantidad de Imágenes', ascending=False)

plt.figure(figsize=(10, 6))
ax = sns.barplot(data=df_counts, y='Comportamiento', x='Cantidad de Imágenes', palette='magma')

# Añadir las cantidades al lado de cada barra para mayor claridad
for p in ax.patches:
    width = p.get_width()
    plt.text(width + 50, p.get_y() + p.get_height()/2. + 0.1, '{:1.0f}'.format(width), ha="left")

plt.title('Distribución de Clases de Comportamiento del Conductor', fontsize=14, fontweight='bold')
plt.xlabel('Número de Imágenes')
plt.ylabel('Tipo de Distracción')
plt.show()

### 2. Visualización de Muestras Aleatorias
Echémosle un vistazo a qué se enfrenta exactamente nuestro modelo (imágenes del mundo real).

In [ ]:
fig, axes = plt.subplots(1, len(classes), figsize=(20, 5))
fig.suptitle('Muestras Aleatorias por Cada Clase', fontsize=16, fontweight='bold', y=1.05)

for idx, c in enumerate(classes):
    class_path = os.path.join(DATA_DIR, c)
    images = [f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    if images:
        random_img = random.choice(images)
        img_path = os.path.join(class_path, random_img)
        
        img = mpimg.imread(img_path)
        axes[idx].imshow(img)
        axes[idx].set_title(c.replace('_', ' ').title())
        axes[idx].axis('off')

plt.tight_layout()
plt.show()

### 3. Informe para el Entregable
*(Puedes usar este texto para nutrir la sección final de tu reporte técnico)*

#### **Análisis de los Tipos de Distracción Más Frecuentes:**
A partir de la distribución de los datos extraídos en este EDA, observamos que gran parte de los comportamientos anómalos se relacionan con el uso del celular, dividiéndose en:
1. **Hablar por teléfono (`talking_phone`):** Elimina una mano del volante y reduce la atención auditiva.
2. **Escribir en el teléfono (`texting_phone`):** Es de los mayores riesgos, pues implica apartar la vista de la vía durante periodos prolongados, además de requerir actividad cognitiva para leer/escribir.
3. **Otras actividades (`other_activities`):** Comprende comportamientos como comer, interactuar con la radio, buscar objetos. Si bien parecen inocuos, desestabilizan el foco en situaciones de frenado de emergencia.
4. **Girarse (`turning`):** Mirar hacia los asientos traseros o hablar con el pasajero de atrás saca completamente el campo de visión del parabrisas.

#### **Posibles Medidas Preventivas (Propuestas):**
Como solución corporativa, proponemos a la empresa de transporte:
- **Alerta Temprana en Cabina (IoT):** Integrar este modelo CNN desarrollado en un sistema embebido (como una Raspberry Pi) dentro del camión. Si detecta la clase `texting_phone` por más de 3 segundos, activar una alarma auditiva.
- **Gamificación y Recompensas:** Los conductores que acumulen mayor porcentaje de tiempo en la clase `safe_driving` durante el mes recibirán bonos económicos, usando la IA como auditor objetivo.
- **Bloqueo de Interfaz:** Integrar un sistema de Bluetooth obligatorio; si la IA detecta el uso físico del teléfono, se emite un reporte inmediato a la central de despacho para control.
- **Pausas Activas:** Si se detecta constante parpadeo o acciones de la categoría `other_activities` (como estirarse, bostezar o comer) el sistema debe recomendar una parada en la próxima zona de descanso.